In [1]:
import numpy as np
import tensorflow as tf
import keras
from datetime import datetime

path_to_file = keras.utils.get_file(
        'shakespeare.txt',
        'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
        )
print(path_to_file)

with open(path_to_file) as f:
    shakespeare_text = f.read()
print(shakespeare_text[:500])

shakespeare_tensor = tf.constant([shakespeare_text])

### Vorbereitung des Datensatzes

In [8]:
vocab = list(set(shakespeare_text.lower().strip()))
print(vocab)
print(f"vocab size = {len(vocab)}")

['z', '\n', 'w', '-', 'b', ':', 'y', 'f', ';', 'm', ',', 'c', '&', '.', ' ', '$', 'o', 'k', 'q', 's', 'i', 'r', '?', 'e', 'u', 'v', 'j', "'", '3', 'd', 'p', 'a', 'n', 'x', '!', 't', 'g', 'h', 'l']
vocab size = 39


In [71]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(char_level=True, lower=True)
tokenizer.fit_on_texts(shakespeare_text.lower())
config = tokenizer.get_config()
sequences = tokenizer.texts_to_sequences(["hello world", "world hello"])
print(sequences)
text = tokenizer.sequences_to_texts(sequences)
print(text)

[[7, 2, 12, 12, 4, 1, 17, 4, 9, 12, 13], [17, 4, 9, 12, 13, 1, 7, 2, 12, 12, 4]]
['h e l l o   w o r l d', 'w o r l d   h e l l o']


In [72]:
n_tokens = len(tokenizer.word_index)
dataset_size = tokenizer.document_count
print(f"n_tokens = {n_tokens}, dataset_size = {dataset_size}")

n_tokens = 39, dataset_size = 1115394


In [73]:
text_array = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1
encoded_1 = text_array[0]
[encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1
set(encoded_1 == encoded)
print(f"{encoded[:100]}")

[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
  0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
  4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24
 17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23
 10 15  3 13]


### Dataset erstellen

In [20]:
train_size = dataset_size * 90 // 100
print(f"train_size = {train_size:,}")

train_size = 1,003,854


In [ ]:
def text_to_dataset(encoded_text, n_tokens, n_steps=50, batch_size=64):
    dataset = tf.data.Dataset.from_tensor_slices(encoded_text)
    dataset = dataset.window(n_steps + 1, shift=1, drop_remainder=True)
    dataset = dataset.flat_map(lambda window_ds: window_ds.batch(n_steps + 1))
    if batch_size > 0:
        dataset = dataset.batch(batch_size)
        dataset = dataset.map(lambda window: (window[:, :-1], window[:, 1:]))
    else:
        dataset = dataset.map(lambda window: (window[:-1], window[1:]))
    dataset = dataset.map(lambda X, y: (tf.one_hot(X, depth=n_tokens), y))
    return dataset.repeat().prefetch(tf.data.AUTOTUNE)


n_steps = 50
batch_size = 64
train_dataset = text_to_dataset(encoded, n_tokens, n_steps=n_steps, batch_size=batch_size)
for X, y in train_dataset.take(1):
    print(f"X = {X.shape}")
    print(f"y = {y.shape}")

### Modell erstellen und trainieren

In [83]:
m1 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(128, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m1.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m1.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_19 (GRU)                    │ (None, None, 128)      │        64,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, None, 39)       │         5,031 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,927 (273.15 KB)

 Trainable params: 69,927 (273.15 KB)

 Non-trainable params: 0 (0.00 B)

In [87]:
model = m1
text = "to be "
tokens = tokenizer.texts_to_sequences([text])
inp_tf = tf.constant(tokens)
inp_tf = tf.one_hot(inp_tf, depth=n_tokens)
print(inp_tf.shape)
y = model.predict(inp_tf)
print(y.shape)

(1, 6, 39)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
(1, 6, 39)


In [88]:
text = "to be o"
tokens = tokenizer.texts_to_sequences([text])
inp_tf = tf.constant(tokens)
inp_tf = tf.one_hot(inp_tf, depth=n_tokens)
print(inp_tf.shape)
y = model.predict(inp_tf)
print(y.shape)
next_token = int(tf.argmax(y[0, -1, :]).numpy() + 1)
print(next_token)
tokens[0].append(next_token)
print(tokens)
tokenizer.sequences_to_texts(tokens)

(1, 7, 39)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
(1, 7, 39)
10
[[3, 4, 1, 22, 2, 1, 4, 10]]


['t o   b e   o n']

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.01, patience=2)
history = m1.fit(
        train_dataset,
        epochs=10,
        steps_per_epoch=train_size // batch_size,
        callbacks=[early_stopping],
        verbose=1
        )
m1.save("../../models/shakespeare_gru_1.keras")

In [89]:
def generate_text(model, tokenizer, prompt, n_next=10):
    tokens = tokenizer.texts_to_sequences([prompt])
    for _ in range(n_next):
        inp_tf = tf.constant(tokens)
        inp_tf = tf.one_hot(inp_tf, depth=n_tokens)
        y = model.predict(inp_tf)
        next_token = int(tf.argmax(y[0, -1, :]).numpy() + 1)
        tokens[0].append(next_token)
    return tokenizer.sequences_to_texts(tokens)[0]


generate_text(m1, tokenizer, "to be o", n_next=20)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


't o   b e   o n w   r   r e   d s e   n t f e o n c o'

In [91]:
m2 = keras.Sequential(
        [
                keras.layers.Input(shape=(None, n_tokens)),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.GRU(64, return_sequences=True),
                keras.layers.Dense(n_tokens, activation="softmax")
                ],
        )
m2.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
m2.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_22 (GRU)                    │ (None, None, 64)       │        20,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_23 (GRU)                    │ (None, None, 64)       │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, None, 39)       │         2,535 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,655 (186.15 KB)

 Trainable params: 47,655 (186.15 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="loss", min_delta=0.01, patience=2)
history = m2.fit(
        train_dataset,
        epochs=10,
        steps_per_epoch=train_size // batch_size,
        callbacks=[early_stopping],
        verbose=1
        )
m2.save("../../models/shakespeare_gru_2L.keras")

In [93]:
generate_text(m2, tokenizer, "to be o", n_next=20)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


't o   b e   o r t u   n w   a n w   d n w   a n w   d'

In [ ]:
m2.evaluate(train_dataset, verbose=1)

 705857/Unknown 5712s 8ms/step - loss: 1.5555